# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadFaizan0023/FlyRank_ML_internship_repo/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: 4 - Logistic Regression
Reason: Handling Multi-class classification task lane as my final output - 'decline_Score' range from (0-5)

**1.1: Import libraries**

In [37]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix

In [38]:
import pandas as pd
import os, getpass
import duckdb

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Time-aware split: latest 90 days window with 60 days training and 30 days testing rows. The training data will not have any client_id or content_id for training but will be included in testing data for testing and evaluation.

**2.1: Read data**

In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
data = pd.read_csv("baseline_action_score_f.csv")

/tmp/ipykernel_1580/3274622562.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("baseline_action_score_f.csv")


In [40]:
data.head(10)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr,engagement_rate,scroll_rate,days_since_update,position_bucket,engagement_bucket,decline_score
0,2026-05-11,client_73cda7b4e4f265ea,content_c036f9d8a7bc0396,18,13.222222,0.0,0.0,0.0,131,11-20,NaN,5
1,2026-04-01,client_9958f0a7ae1df715,content_d06983da58555f50,8,10.000000,0.0,0.0,0.0,38,4-10,NaN,5
2,2026-04-01,client_9958f0a7ae1df715,content_e9d1789ebf0ff1f6,1,10.000000,0.0,0.0,0.0,38,4-10,NaN,5
3,2026-04-01,client_9958f0a7ae1df715,content_70f2cea7bfc5dfb6,4,68.250000,0.0,0.0,0.0,49,21-100,NaN,5
4,2026-04-01,client_9958f0a7ae1df715,content_cd046a34819b45af,5,41.200000,0.0,0.0,0.0,49,21-100,NaN,5
5,2026-04-01,client_9958f0a7ae1df715,content_8acbcd49af928119,4,37.750000,0.0,0.0,0.0,49,21-100,NaN,5
6,2026-05-11,client_73cda7b4e4f265ea,content_d01758ff6fff5197,16,29.625000,0.0,0.0,0.0,131,21-100,NaN,5
7,2026-05-11,client_73cda7b4e4f265ea,content_02f950076e61df04,16,12.750000,0.0,0.0,0.0,47,11-20,NaN,5
8,2026-04-01,client_9958f0a7ae1df715,content_d8288f22519f7d53,12,26.250000,0.0,0.0,0.0,38,21-100,NaN,5
9,2026-04-01,client_9958f0a7ae1df715,content_c040204537f06a59,67,31.955224,0.0,0.0,0.0,38,21-100,NaN,5


In [41]:
data.tail(10)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr,engagement_rate,scroll_rate,days_since_update,position_bucket,engagement_bucket,decline_score
1683287,2026-05-30,client_1a730cb2640a1abf,content_63bc11c45c14e8fe,1720,7.182558,0.000581,0.250000,0.250000,26,4-10,low,0
1683288,2026-06-28,client_73cda7b4e4f265ea,content_c2799e6502d2de3f,132,5.537879,0.007576,1.000000,1.000000,1,4-10,very_high,0
1683289,2026-04-24,client_23a62021009f63c4,content_138abe69ee5bfd14,924,7.462121,0.009740,0.083333,0.040000,14,4-10,low,0
1683290,2026-04-01,client_3f0ce4d44fe94f3d,content_b2347ac65344f8c0,446,1.914798,0.002242,1.000000,1.000000,12,1-3,very_high,0
1683291,2026-05-30,client_1a730cb2640a1abf,content_f233334aec6c9b56,2438,2.150123,0.001641,0.111111,0.111111,26,1-3,low,0
1683292,2026-05-30,client_1a730cb2640a1abf,content_54c0fa31d867a8ea,281,4.209964,0.010676,0.200000,0.200000,26,4-10,low,0
1683293,2026-05-30,client_1a730cb2640a1abf,content_d9ff67b581e72d74,176,7.573864,0.011364,0.500000,0.500000,26,4-10,medium,0
1683294,2026-04-24,client_23a62021009f63c4,content_dd7c5f83b5b50ee6,115,8.878261,0.026087,0.500000,0.250000,14,4-10,medium,0
1683295,2026-05-15,client_73cda7b4e4f265ea,content_2fe75239353ca721,710,3.773239,0.005634,0.166667,0.375000,7,4-10,low,0
1683296,2026-06-25,client_73cda7b4e4f265ea,content_381e41a7ae8bfb12,345,5.666667,0.002899,1.000000,1.000000,8,4-10,very_high,0


**2.2: Handle missing values**

In [42]:
data.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions',
       'gsc_avg_position', 'ctr', 'engagement_rate', 'scroll_rate',
       'days_since_update', 'position_bucket', 'engagement_bucket',
       'decline_score'],
      dtype='object')

In [43]:
data['gsc_avg_position'].isnull().sum()

np.int64(8)

In [44]:
data['ctr'].isnull().sum()

np.int64(0)

In [45]:
data['engagement_rate'].isnull().sum()

np.int64(31489)

In [46]:
data['scroll_rate'].isnull().sum()

np.int64(4229)

In [47]:
data['days_since_update'].isnull().sum()

np.int64(0)

In [48]:
data['decline_score'].isnull().sum()

np.int64(0)

In [49]:
data = data.dropna(subset=['gsc_avg_position']).reset_index(drop=True)

In [50]:
data = data.dropna(subset=['engagement_rate']).reset_index(drop=True)

In [51]:
data = data.dropna(subset=['scroll_rate']).reset_index(drop=True)

In [52]:
data['gsc_avg_position'].isnull().sum()

np.int64(0)

In [53]:
data['engagement_rate'].isnull().sum()

np.int64(0)

In [54]:
data['scroll_rate'].isnull().sum()

np.int64(0)

**2.3: Train-Test split**

In [55]:
data['report_date'] = pd.to_datetime(data['report_date'])

train_data = data[(data['report_date'] >= '2026-04-01') & (data['report_date'] <= '2026-05-30')].copy()

test_data = data[(data['report_date'] >= '2026-06-01') & (data['report_date'] <= '2026-06-30')].copy()

print(f"Training set size: {len(train_data)}")
print(f"Testing set size: {len(test_data)}")
display(train_data.head())
display(test_data.head())

Training set size: 1092243
Testing set size: 539246


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr,engagement_rate,scroll_rate,days_since_update,position_bucket,engagement_bucket,decline_score
0,2026-05-11,client_73cda7b4e4f265ea,content_c036f9d8a7bc0396,18,13.222222,0.0,0.0,0.0,131,11-20,NaN,5
1,2026-04-01,client_9958f0a7ae1df715,content_d06983da58555f50,8,10.000000,0.0,0.0,0.0,38,4-10,NaN,5
2,2026-04-01,client_9958f0a7ae1df715,content_e9d1789ebf0ff1f6,1,10.000000,0.0,0.0,0.0,38,4-10,NaN,5
3,2026-04-01,client_9958f0a7ae1df715,content_70f2cea7bfc5dfb6,4,68.250000,0.0,0.0,0.0,49,21-100,NaN,5
4,2026-04-01,client_9958f0a7ae1df715,content_cd046a34819b45af,5,41.200000,0.0,0.0,0.0,49,21-100,NaN,5


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr,engagement_rate,scroll_rate,days_since_update,position_bucket,engagement_bucket,decline_score
76,2026-06-30,client_a22068e339bf95f5,content_4226eaa46864a67f,15,19.333333,0.0,0.0,0.0,47,11-20,NaN,5
132,2026-06-30,client_7de9989c909e91a5,content_61596e2dda4e4008,8,14.125000,0.0,0.0,0.0,35,11-20,NaN,5
133,2026-06-30,client_7de9989c909e91a5,content_ddd4062b6401587c,12,21.500000,0.0,0.0,0.0,35,21-100,NaN,5
134,2026-06-30,client_7de9989c909e91a5,content_6d9e6b27becbe515,16,18.562500,0.0,0.0,0.0,35,11-20,NaN,5
135,2026-06-30,client_7de9989c909e91a5,content_a4ed21325c168ae7,1,10.000000,0.0,0.0,0.0,35,4-10,NaN,5


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**3.1: Train X-y split**

In [56]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
X_train = train_data.drop(columns=['report_date', 'client_hash_id', 'content_hash_id','position_bucket', 'engagement_bucket', 'decline_score'], axis=1)
y_train = train_data['decline_score']

In [57]:
X_train.columns

Index(['gsc_impressions', 'gsc_avg_position', 'ctr', 'engagement_rate',
       'scroll_rate', 'days_since_update'],
      dtype='object')

In [58]:
print(len(X_train))
print(len(y_train))

1092243
1092243


**3.2: Test X-y split**

In [59]:
X_test = test_data.drop(columns=['report_date', 'client_hash_id', 'content_hash_id','position_bucket', 'engagement_bucket','decline_score'], axis=1)
y_test = test_data['decline_score']

In [60]:
X_test.columns

Index(['gsc_impressions', 'gsc_avg_position', 'ctr', 'engagement_rate',
       'scroll_rate', 'days_since_update'],
      dtype='object')

In [61]:
print(len(X_test))
print(len(y_test))

539246
539246


**3.3: Model: LogisticRegression**

**3.3.1: Model training**

In [62]:
model1 = LogisticRegression(max_iter=1000, class_weight='balanced', multi_class='multinomial')
model1.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(class_weight='balanced', max_iter=1000,
                   multi_class='multinomial')

**3.3.2: Model predictions and evaluation**

In [63]:
y_pred = model1.predict(X_test)
y_proba = model1.predict_proba(X_test)


In [64]:
f1 = f1_score(y_test, y_pred, average='macro')
auc = roc_auc_score(y_test, y_proba, multi_class='ovo')

print(f"F1: {f1:.4f}")
print(f"ROC-AUC: {auc:.4f}")

F1: 0.4445
ROC-AUC: 0.8603


**3.3.3: Comparison table**

In [65]:
print("Actual (y_test) vs Predicted (y_pred):")
comparison_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
display(comparison_df.head(20))
display(comparison_df.tail(20))
print(f"Total rows: {len(comparison_df)}\n")

Actual (y_test) vs Predicted (y_pred):


,Actual,Predicted
76,5,4
132,5,4
133,5,4
134,5,4
135,5,4
136,5,5
181,5,4
198,5,5
199,5,5
211,5,5


,Actual,Predicted
1648475,0,0
1648478,0,1
1648479,0,0
1648480,0,0
1648481,0,0
1648482,0,0
1648483,0,0
1648484,0,0
1648486,0,0
1648487,0,0


Total rows: 539246



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [66]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

**4.1: Confusion matrix**

In [67]:
print(confusion_matrix(y_test, y_pred))

[[ 7162  1187  1225     5     1     0]
 [ 6961  9849  3102   887   715   382]
 [ 8553 15456 79132  7014  4007  1855]
 [  726 10631 54926 57562 31304 19043]
 [    5  1533 12214 27707 56209 61378]
 [    0     2     1   721 15276 42515]]


**4.2: Classification report**

In [68]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.31      0.75      0.43      9580
           1       0.25      0.45      0.33     21896
           2       0.53      0.68      0.59    116017
           3       0.61      0.33      0.43    174192
           4       0.52      0.35      0.42    159046
           5       0.34      0.73      0.46     58515

    accuracy                           0.47    539246
   macro avg       0.43      0.55      0.44    539246
weighted avg       0.52      0.47      0.46    539246



**Interpretation**

The classifier shows a clear precision-recall tradeoff split by class position. Classes 0 and 5 (the extremes, have high recall (75%, 73%) but low precision (31%, 34%): the model over-predicts these classes, catching most true cases but also misfiring on many that actually belong to a neighboring class. Classes 3 and 4 show the opposite pattern — decent precision (61%, 52%) but weak recall (33%, 35%), meaning when the model predicts these classes it's often right, but it fails to identify most of the true cases, which get pulled toward the extremes instead. Support is heavily skewed toward classes 2-4 (450K of 539K rows), so the 0.47 accuracy and 0.46 weighted F1 are dominated by mid-range performance, while the 0.44 macro F1 reveals the smaller classes (0, 1) are proportionally weaker than the headline numbers suggest. Combined with the confusion matrix, this indicates the model treats decline severity as a continuum it can rank (consistent with the 0.86 ROC-AUC) but linear decision boundaries aren't sharp enough to separate five internal cutoffs — it's most confident and correct only at the two ends of the scale.

## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.